# Vector Search, end to end: kNN → ANN → IVF-PQ → HNSW

**Companion notebook for the 1-hour tech talk.**

Every claim in the talk is *measured here*, on a real index, on this machine. Nothing is
hard-coded — if you change the dataset size at the top, every table and chart re-computes.

**What you'll build, in order:**

| Section | Technique | The value it adds |
|---|---|---|
| 1 | Exact kNN | Correct answers, zero infrastructure — and your **ruler** |
| 2 | Recall@k | The one number that makes "approximate" measurable |
| 3 | **IVF** | Skip ~99% of the data |
| 4 | **PQ** | 32× less memory |
| 5 | **Rerank** | Buy back the recall PQ lost, for almost free |
| 6 | **HNSW** | Best recall-per-millisecond, tunable *per query* |
| 7 | The ledger | All of it on one scoreboard |
| 8 | Filtered search | The gotcha that actually decides your architecture |

**How to use it:** run the cells top to bottom (Runtime → Run all). Read the note above each
cell first — the notes are the talk. Total runtime is roughly **4–6 minutes** on a free Colab CPU.

_Save a copy to your Drive (File → Save a copy in Drive) before editing._

In [ ]:
# Setup. faiss is Facebook AI Similarity Search — the reference library for this whole topic.
# It ships prebuilt wheels, so this is fast and won't compile anything.
!pip install -q faiss-cpu

In [ ]:
import os
import time

import numpy as np
import faiss
import matplotlib.pyplot as plt

print("faiss version:", faiss.__version__)
print("CPU cores available:", os.cpu_count())

N_BUILD_THREADS = min(4, os.cpu_count() or 1)

---
## 0. The problem: "SupportBot"

One concrete problem carries the whole notebook, so every technique is scored on the same task.

> We have **200,000** past support tickets. A new ticket arrives; surface the most similar past
> tickets so the agent can reuse the resolution. Each ticket is embedded as a **256-dimensional**
> float32 vector. We need **200 queries/sec** at **p99 under 50 ms**.

Everything below is measured against that. Change `N_TICKETS` and re-run to see how the story
shifts — that's the most instructive edit you can make.

In [ ]:
# ── The problem definition. Change these and re-run everything. ──────────────
N_TICKETS  = 200_000    # how many past tickets are in the corpus
N_DIMS     = 256        # embedding dimension
N_QUERIES  = 1_000      # how many queries we benchmark with
K          = 10         # we want the top-10 most similar tickets

TARGET_QPS = 200        # the throughput the service must sustain

# ── Shape of the synthetic corpus (explained in the next cell) ───────────────
N_TOPICS = 60           # how many latent "topics" the tickets cluster around
SPREAD   = 1.0          # how strongly a ticket is pulled toward its topic
ALPHA    = 1.0          # how fast the embedding's eigenvalue spectrum decays

raw_bytes = N_TICKETS * N_DIMS * 4
print(f"Raw corpus: {N_TICKETS:,} x {N_DIMS} x 4 bytes = {raw_bytes/1e6:.1f} MB")

### Step 1 — Build a corpus that behaves like real embeddings

We generate vectors rather than downloading a dataset so the notebook runs anywhere in seconds.
But **random Gaussian noise would be badly misleading** — it's the pathological worst case for
every technique here, and it would make ANN look far worse than it is in practice.

Real embeddings have two properties we reproduce deliberately:

1. **They cluster by topic.** Tickets about billing land near other billing tickets. We draw
   `N_TOPICS` centers and scatter tickets around them.
2. **Their eigenvalue spectrum decays.** A few directions carry most of the variance;
   dimensions are strongly correlated, not independent. We build that in with a decaying
   spectrum `eigenvalues[i] ∝ 1/(1+i)^ALPHA` applied in a random rotated basis.

**Property 2 is the one people miss, and it's why PQ works at all.** Compression exploits
correlation between dimensions. On white noise there is none to exploit, so PQ collapses. On
real embeddings there's plenty. Skip this detail and your synthetic benchmark will tell you
lies about PQ.

Finally we **L2-normalize** every vector. That makes cosine similarity, inner product, and
Euclidean distance all rank identically, so we can use whichever is convenient without
changing a single result.

In [ ]:
rng = np.random.default_rng(0)

# 1. A decaying eigenvalue spectrum — the signature of real embedding data.
eigenvalues = (1.0 / (1.0 + np.arange(N_DIMS))) ** ALPHA
eigenvalues = eigenvalues / eigenvalues.sum() * N_DIMS      # rescale to keep total variance = N_DIMS
eigenvalues = eigenvalues.astype(np.float32)

# 2. A random orthonormal basis, so the strong directions aren't axis-aligned
#    (real embeddings have no reason to line up with coordinate axes).
rotation, _ = np.linalg.qr(rng.normal(size=(N_DIMS, N_DIMS)))
rotation = rotation.astype(np.float32)

def draw(n_samples):
    """Draw n_samples vectors with the spectrum + rotation defined above."""
    z = rng.normal(size=(n_samples, N_DIMS)).astype(np.float32)
    z = z * np.sqrt(eigenvalues)          # stretch each direction by its eigenvalue
    return z @ rotation.T                 # rotate out of the axis-aligned frame

print("drawing topic centers ...")
topic_centers = draw(N_TOPICS)

In [ ]:
# 3. Each ticket = (its topic center * SPREAD) + its own personal variation.
n_total = N_TICKETS + N_QUERIES
topic_of = rng.integers(0, N_TOPICS, size=n_total)

everything = SPREAD * topic_centers[topic_of] + draw(n_total)

# 4. Normalize to unit length, so cosine == inner product == (monotone in) L2.
norms = np.linalg.norm(everything, axis=1, keepdims=True)
everything = everything / norms

# 5. Split into the corpus we index, and the queries we benchmark with.
tickets = np.ascontiguousarray(everything[:N_TICKETS], dtype=np.float32)
queries = np.ascontiguousarray(everything[N_TICKETS:], dtype=np.float32)

del everything
print("tickets:", tickets.shape, " queries:", queries.shape)

### Step 2 — Look at the data before using it

Never index a corpus you haven't eyeballed. Three sanity checks:
norms should be exactly 1, a ticket should be much more similar to its *own* topic-mates than to
a random ticket, and the spectrum should be visibly skewed.

In [ ]:
print(f"memory        : {tickets.nbytes/1e6:.1f} MB")
print(f"norm of row 0 : {np.linalg.norm(tickets[0]):.6f}   (should be 1.0)")
print(f"first 6 dims  : {np.round(tickets[0][:6], 4)}")

# How similar is a ticket to a same-topic ticket vs. a random ticket?
same_topic = np.where(topic_of[:N_TICKETS] == topic_of[0])[0][1:200]
random_ids = rng.integers(0, N_TICKETS, size=200)

print(f"\nmean cosine to SAME-topic tickets  : {float(np.mean(tickets[same_topic] @ tickets[0])):.3f}")
print(f"mean cosine to RANDOM tickets      : {float(np.mean(tickets[random_ids] @ tickets[0])):.3f}")

In [ ]:
# The eigenvalue spectrum: a few directions carry most of the variance.
sample = tickets[:20_000]
centered = sample - sample.mean(axis=0)
singular_values = np.linalg.svd(centered, compute_uv=False)
variance = singular_values ** 2
variance = variance / variance.sum()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(variance[:64], marker="o", ms=3)
ax[0].set_title("Variance per direction (top 64)")
ax[0].set_xlabel("direction")
ax[0].set_ylabel("share of variance")

ax[1].plot(np.cumsum(variance))
ax[1].axhline(0.9, ls="--", c="crimson", lw=1)
ax[1].set_title("Cumulative variance")
ax[1].set_xlabel("number of directions")
ax[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

n_for_90 = int(np.searchsorted(np.cumsum(variance), 0.90)) + 1
print(f"{n_for_90} of {N_DIMS} directions carry 90% of the variance.")
print("That redundancy is exactly what PQ will compress in section 4.")

---
# 1. Exact kNN

> ### Value: it is **correct**, and it costs you nothing to build.
>
> Two separate values, and the second is the one people forget.
> **(a)** It's the simplest thing that works — no index, no training, no tuning, no new service.
> **(b)** It's your **ruler**. Every approximate method below is scored against it. Without it,
> you have no way to detect that your fancy index silently got worse.

### Step 1 — The entire search engine, in three lines

In [ ]:
def exact_search(query, k=K):
    scores = tickets @ query                      # cosine similarity to every ticket
    top = np.argpartition(-scores, k)[:k]         # k best, unordered — cheaper than a full sort
    return top[np.argsort(-scores[top])]          # now order just those k

result = exact_search(queries[0])
print("top-10 ticket ids :", result)
print("their similarities:", np.round(tickets[result] @ queries[0], 3))

That is a complete, production-viable search engine — **for a small enough corpus**.

**If your team is standing up a vector database for 50,000 rows, this cell is the whole talk.**
It's exact, it has zero operational surface area, adding a vector is `np.append`, and there is
no index to rebuild, tune, or monitor.

### Step 2 — Compute ground truth (the ruler)

We run exact search for all 1,000 benchmark queries **once** and keep the answers. From here on,
"recall" means "agreement with this array."

In [ ]:
t0 = time.perf_counter()

similarity = queries @ tickets.T                          # (N_QUERIES, N_TICKETS)
part = np.argpartition(-similarity, K, axis=1)[:, :K]     # top-K per row, unordered
rows = np.arange(N_QUERIES)[:, None]
order = np.argsort(-similarity[rows, part], axis=1)       # sort within those K
ground_truth = part[rows, order]

del similarity                                            # this matrix is large - free it now
print(f"ground truth {ground_truth.shape} computed in {time.perf_counter()-t0:.2f}s")

### Step 3 — Time one query, then price the fleet

This is the number that motivates everything after it. We time a **single** query at a time,
because that's what a serving system actually does — not a big batched matrix multiply.

In [ ]:
n_timed = 100
t0 = time.perf_counter()
for i in range(n_timed):
    exact_search(queries[i])
EXACT_MS = (time.perf_counter() - t0) / n_timed * 1000

cores_needed = TARGET_QPS * EXACT_MS / 1000

print(f"exact search      : {EXACT_MS:.2f} ms per query")
print(f"to sustain {TARGET_QPS} QPS : {cores_needed:.1f} CPU cores, saturated, forever")
print()
print("And the cost is LINEAR in corpus size:")
for multiple in [1, 10, 100]:
    print(f"  {N_TICKETS*multiple:>12,} tickets -> {EXACT_MS*multiple:8.1f} ms/query"
          f" -> {cores_needed*multiple:7.1f} cores")

**That last block is the argument for the rest of the notebook.** Exact search doesn't fail —
it just gets linearly more expensive until the bill is absurd. Ten million tickets is a rack of
machines to answer a question a laptop should handle.

And there's **no exact way out**. kd-trees and R-trees genuinely work in 2–3 dimensions and are
useless at 256: above roughly 20 dimensions, nearly every point sits at nearly the same distance
from your query, so there's no tight region to prune. That's the *curse of dimensionality*, and
it's the reason this entire field exists.

---
# 2. Recall@k — making "approximate" measurable

> ### Value: it converts a vague worry ("are we losing results?") into one number you can put on a dashboard.

**Recall@k** = the fraction of the true top-k that the index actually returned.
Return 9 of the true top 10 → recall@10 = 0.9. That's it.

The important framing for skeptics in the room: **your embeddings were already approximate.**
The true #7 nearest neighbor by cosine distance isn't a ground truth about ticket relevance —
it's one model's opinion. Missing it usually costs nothing a user can perceive.

But be honest about the exception: if you're doing **dedup or exact retrieval**, where a missed
neighbor is a correctness bug rather than a ranking nudge, recall 0.95 means **5% wrong answers**.
Know which regime you're in.

### The three formulas behind this whole notebook

**1. What we're asking for.** The true top-$k$ for query $q$ over corpus $X$:

$$\mathrm{TopK}(q) \;=\; \underset{S \subseteq X,\; |S| = k}{\arg\min} \; \sum_{x \in S} \lVert q - x \rVert^2$$

**2. How we grade an approximation.** Given the approximate result $\widehat{\mathrm{TopK}}(q)$
over $Q$ benchmark queries:

$$\text{recall@}k \;=\; \frac{1}{|Q|\,k} \sum_{q \in Q} \bigl| \mathrm{TopK}(q) \cap \widehat{\mathrm{TopK}}(q) \bigr|$$

**3. Why PQ can skip the multiplications.** Split $q$ and $x$ into $m$ chunks. Squared distance is
*separable* — it's just a sum over chunks:

$$\lVert q - x \rVert^2 \;=\; \sum_{i=1}^{m} \lVert q^{(i)} - x^{(i)} \rVert^2 \;\approx\; \sum_{i=1}^{m} \underbrace{\lVert q^{(i)} - c^{(i)}_{\,\text{code}_i(x)} \rVert^2}_{\text{precomputed: a table lookup}}$$

**That separability is the entire trick.** Because the total is a sum of independent per-chunk
terms, and each chunk has only 256 possible values, every term can be looked up instead of
computed. Section 4 builds this table by hand.

In [ ]:
def recall_at_k(found, k=K):
    """Fraction of the true top-k that `found` actually contains."""
    n = len(found)
    hits = 0
    for i in range(n):
        hits += len(set(found[i][:k]) & set(ground_truth[i][:k]))
    return hits / (n * k)

# Sanity check: exact search must score exactly 1.0 against itself.
# If this ever fails, the ruler is broken and every number below is meaningless.
check = np.stack([exact_search(queries[i]) for i in range(100)])
check_recall = recall_at_k(check)

assert check_recall == 1.0, f"ground truth disagrees with exact search: {check_recall}"
print("recall of exact search vs ground truth:", check_recall, " <- the ruler is calibrated")

In [ ]:
# All search benchmarks below run SINGLE-THREADED so latencies are comparable.
# (Index *building* still uses several threads — that's not what we're measuring.)
faiss.omp_set_num_threads(1)

def bench(search_fn, n=N_QUERIES, label=""):
    """Time a search function over n queries and score its recall."""
    t0 = time.perf_counter()
    found = search_fn(queries[:n])
    ms = (time.perf_counter() - t0) / n * 1000
    rec = recall_at_k(found[:n])
    if label:
        print(f"{label:<28} {ms:7.3f} ms/query   recall@{K} {rec:.3f}")
    return ms, rec

---
# 3. IVF — Inverted File Index

> ### Value: you stop looking at data that was never going to win.
> **~50× less work for a few points of recall.**

**The library analogy:** you don't scan every book in the building. You walk to the right few
shelves and scan those.

**Build:** run k-means over the corpus to get `nlist` centroids. Every ticket is assigned to its
nearest centroid, forming `nlist` buckets (Voronoi cells).

**Search:** compare the query against the `nlist` centroids only (cheap), pick the `nprobe`
closest buckets, and brute-force *just those*.

In [ ]:
NLIST = 512      # number of buckets. Rule of thumb: ~sqrt(N_TICKETS), rounded to something tidy.

faiss.omp_set_num_threads(N_BUILD_THREADS)

quantizer = faiss.IndexFlatL2(N_DIMS)                                  # holds the centroids
ivf = faiss.IndexIVFFlat(quantizer, N_DIMS, NLIST, faiss.METRIC_L2)

t0 = time.perf_counter()
ivf.train(tickets)      # k-means: this is the step people forget IVF needs
ivf.add(tickets)        # assign every ticket to its bucket
print(f"IVF built in {time.perf_counter()-t0:.1f}s   ({NLIST} buckets,"
      f" ~{N_TICKETS//NLIST:,} tickets each)")

faiss.omp_set_num_threads(1)

### The `nprobe` sweep — the most important table in this section

`nprobe` is the dial. Watch **two** things: how fast recall climbs, and how fast it stops climbing.

In [ ]:
ivf_sweep = []

for nprobe in [1, 2, 4, 8, 16, 32, 64, NLIST]:
    ivf.nprobe = nprobe
    ms, rec = bench(lambda Q: ivf.search(Q, K)[1])
    scanned = N_TICKETS * nprobe / NLIST
    ivf_sweep.append({"nprobe": nprobe, "scanned": scanned, "ms": ms, "recall": rec})

print(f"{'nprobe':>7} {'scanned':>12} {'% of corpus':>12} {'ms/query':>10} {'recall@10':>11}")
print("-" * 56)
for r in ivf_sweep:
    print(f"{r['nprobe']:>7} {r['scanned']:>12,.0f} {100*r['scanned']/N_TICKETS:>11.1f}%"
          f" {r['ms']:>10.3f} {r['recall']:>11.3f}")

print(f"\nexact search for comparison: {EXACT_MS:.3f} ms, recall 1.000")

**Two things to point at in that table:**

1. **One row is your product decision.** Not a benchmark, not a library default — you pick the
   row that fits your latency budget. That is the actual deliverable of index tuning.
2. **Recall climbs fast, then flattens, while cost keeps climbing linearly.** Almost all of the
   value is in the first few probes. Doubling `nprobe` from 32 to 64 costs you double the time
   and buys approximately nothing.

The chart makes the diminishing return impossible to miss.

In [ ]:
probes = [r["nprobe"] for r in ivf_sweep]
recalls = [r["recall"] for r in ivf_sweep]
latencies = [r["ms"] for r in ivf_sweep]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].plot(probes, recalls, marker="o")
ax[0].set_xscale("log")
ax[0].set_xlabel("nprobe")
ax[0].set_ylabel("recall@10")
ax[0].set_title("Recall saturates quickly")
ax[0].grid(alpha=.3)

ax[1].plot(latencies, recalls, marker="o")
ax[1].axvline(EXACT_MS, ls="--", c="crimson", lw=1)
ax[1].text(EXACT_MS, 0.5, " exact", c="crimson", rotation=90, va="center", fontsize=8)
ax[1].set_xscale("log")
ax[1].set_xlabel("ms per query")
ax[1].set_ylabel("recall@10")
ax[1].set_title("The trade-off curve (up and to the LEFT is better)")
ax[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

### Where IVF's errors come from — the boundary problem

IVF's failure mode is **specific and predictable**: a query near a cell boundary has its true
nearest neighbor sitting just across the line, in a cell we never probed.

That predictability is genuinely useful. **Errors you can explain are errors you can debug** —
and `nprobe` is exactly the fix. Let's find a real example in 2D and look at it.

In [ ]:
# A 2D toy corpus so we can actually see the cells.
pts = rng.normal(size=(3000, 2)).astype(np.float32)

kmeans = faiss.Kmeans(2, 12, niter=25, seed=1, verbose=False)
kmeans.train(pts)
centers2d = kmeans.centroids
_, assign2d = kmeans.index.search(pts, 1)
assign2d = assign2d.ravel()

# Hunt for a query whose true nearest neighbour lives in a DIFFERENT cell.
victim_q = None
for _ in range(4000):
    q = rng.normal(size=(1, 2)).astype(np.float32)
    true_nn = int(np.argmin(((pts - q) ** 2).sum(axis=1)))
    _, q_cell = kmeans.index.search(q, 1)
    if assign2d[true_nn] != q_cell[0, 0]:
        victim_q, victim_nn, victim_cell = q[0], true_nn, int(q_cell[0, 0])
        break

print("query cell:", victim_cell, " but its true nearest neighbour lives in cell:",
      assign2d[victim_nn])

In [ ]:
plt.figure(figsize=(6.5, 6))
plt.scatter(pts[:, 0], pts[:, 1], c=assign2d, cmap="tab20", s=6, alpha=.45)
plt.scatter(centers2d[:, 0], centers2d[:, 1], c="black", marker="x", s=90, label="centroids")

probed = pts[assign2d == victim_cell]
plt.scatter(probed[:, 0], probed[:, 1], facecolors="none", edgecolors="black",
            s=18, lw=.4, label=f"cell we probe (nprobe=1)")

plt.scatter(*victim_q, c="red", marker="*", s=420, zorder=5, label="query")
plt.scatter(*pts[victim_nn], c="lime", marker="D", s=140, edgecolors="black",
            zorder=5, label="TRUE nearest neighbour (missed!)")
plt.plot([victim_q[0], pts[victim_nn][0]], [victim_q[1], pts[victim_nn][1]],
         "r--", lw=1.2)

plt.legend(loc="upper left", fontsize=8)
plt.title("IVF's failure mode: the answer is one cell over")
plt.tight_layout()
plt.show()

**That picture is the entire intuition for `nprobe`.** The green diamond is genuinely the closest
point, but it fell on the far side of a boundary. Probing 2 or 3 cells instead of 1 catches it.

### Where IVF's value runs out

- **Memory is completely unchanged.** IVF still stores all 200 MB of full vectors. It buys
  **time**, not **space**. That's the setup for the next section.
- **It needs training.** k-means over a representative sample before you can insert anything.
- **Distribution drift degrades it.** Centroids fitted on last year's tickets get lopsided as
  topics shift; one bucket ends up with 10× its share and probing it is slow. Retraining is
  real operational work.

---
# 4. PQ — Product Quantization

> ### Value: ~32× compression, which changes what hardware you need.
> This is a **deployment** value, not a latency value. PQ is what turns "we need a dedicated
> cluster with a lot of RAM" into "this fits next to the app."

**The palette analogy:** instead of storing every pixel's exact color, store an index into a
256-color palette.

**The construction:**

1. **Split** each 256-dim vector into `m = 32` sub-vectors of 8 dims each.
2. **Cluster each chunk position independently** — 256 centroids over all the chunk-1's, another
   256 over all the chunk-2's, and so on. 32 small codebooks.
3. **Encode** each chunk as the id of its nearest centroid: a number 0–255, i.e. **one byte**.

**1024 bytes → 32 bytes.**

*"Product" because the effective codebook is the Cartesian product of the 32 sub-codebooks —
256³² representable vectors from only 32×256 stored centroids. That's the trick that makes it
tractable.*

In [ ]:
M_SUB = 32          # 256 dims / 32 sub-vectors = 8 dims each
N_BITS = 8          # 8 bits -> 256 centroids per sub-codebook -> exactly 1 byte per chunk

# Training on a sample is standard practice and much faster than training on everything.
TRAIN_SAMPLE = np.ascontiguousarray(tickets[:30_000])

faiss.omp_set_num_threads(N_BUILD_THREADS)

pq = faiss.IndexPQ(N_DIMS, M_SUB, N_BITS)
pq.pq.cp.niter = 6            # k-means iterations - 6 is plenty for a demo

t0 = time.perf_counter()
pq.train(TRAIN_SAMPLE)
pq.add(tickets)
print(f"PQ trained + encoded in {time.perf_counter()-t0:.1f}s")

faiss.omp_set_num_threads(1)

In [ ]:
# Look at what a ticket has actually become: 32 bytes.
codes = faiss.vector_to_array(pq.codes).reshape(N_TICKETS, M_SUB)

print("codes dtype :", codes.dtype, " shape:", codes.shape)
print("ticket 0 was:", np.round(tickets[0][:8], 4), "... (256 floats)")
print("ticket 0 is :", codes[0], f"({M_SUB} bytes)")

pq_bytes = codes.nbytes
print(f"\ncorpus: {raw_bytes/1e6:.1f} MB  ->  {pq_bytes/1e6:.1f} MB"
      f"   ({raw_bytes/pq_bytes:.0f}x smaller)")

# How much did we lose? Reconstruct ticket 0 from its 32 bytes.
reconstructed = pq.reconstruct(0)
fidelity = float(reconstructed @ tickets[0] / np.linalg.norm(reconstructed))
print(f"cosine(original, reconstructed-from-32-bytes) = {fidelity:.4f}")

### The instance-type slide

Three separate arguments live in this table, and they get better as you go down.

In [ ]:
print(f"{'corpus size':>16} {'raw (float32)':>16} {'with PQ':>12}   what changes")
print("-" * 78)
for n, note in [(N_TICKETS,      "fits INSIDE your app process — no separate service"),
                (N_TICKETS * 10, "still a normal box - growth stopped being an infra project"),
                (100_000_000,    "one big machine instead of a cluster"),
                (1_000_000_000,  "billion-scale on a single host — this row is why PQ exists")]:
    raw_gb = n * N_DIMS * 4 / 1e9
    pq_gb = n * M_SUB / 1e9
    print(f"{n:>16,} {raw_gb:>13.1f} GB {pq_gb:>9.2f} GB   {note}")

### The compression dial has a cliff (unlike `nprobe`)

`m` — the number of sub-vectors — is PQ's dial. More sub-vectors = finer chunks = better
fidelity = more bytes. Unlike IVF's smooth `nprobe` curve, **this one falls off a cliff**.

In [ ]:
faiss.omp_set_num_threads(N_BUILD_THREADS)
pq_sweep = []

for m in [16, 32, 64, 128]:
    idx = faiss.IndexPQ(N_DIMS, m, N_BITS)
    idx.pq.cp.niter = 6
    idx.train(TRAIN_SAMPLE)
    idx.add(tickets)

    faiss.omp_set_num_threads(1)
    ms, rec = bench(lambda Q: idx.search(Q, K)[1], n=500)
    faiss.omp_set_num_threads(N_BUILD_THREADS)

    pq_sweep.append({"m": m, "bytes": m, "ratio": (N_DIMS * 4) / m, "ms": ms, "recall": rec})
    del idx

faiss.omp_set_num_threads(1)

print(f"{'m':>5} {'bytes/vec':>11} {'compression':>13} {'ms/query':>10} {'recall@10':>11}")
print("-" * 54)
for r in pq_sweep:
    print(f"{r['m']:>5} {r['bytes']:>11} {r['ratio']:>12.0f}x {r['ms']:>10.3f} {r['recall']:>11.3f}")

**Read those recall numbers and don't panic** — they look alarming on purpose. Raw PQ ranking is
genuinely noisy, because at 32× compression the reconstruction error is comparable to the gap
between the true #1 and the true #40. Section 5 fixes almost all of it with three lines of code.

The real lesson here: **PQ alone is not a search index — it's a *shortlisting* index.**

### The clever part: you never decompress

The natural assumption is that compressed data must be decompressed before you can use it.
**PQ never does.** Here's the trick, built by hand so you can see there's nothing hidden.

At query time, split the query into its 32 chunks and precompute a **lookup table**: the distance
from query-chunk *i* to each of the 256 centroids in codebook *i*. That's 32×256 = 8,192 tiny
computations, done **once per query**.

Then the distance to **any** ticket in the corpus is: **32 table lookups and 32 adds.**
No multiplications. No 256-dimensional math.

**The same 32 bytes that saved memory also made the comparison cheaper.** Compression and speed
normally trade against each other; here one bought both.

In [ ]:
# Pull the learned codebooks out of faiss: (M_SUB, 256, dims_per_chunk)
codebooks = faiss.vector_to_array(pq.pq.centroids).reshape(M_SUB, 2 ** N_BITS, N_DIMS // M_SUB)
print("codebooks shape:", codebooks.shape)

q = queries[0]
q_chunks = q.reshape(M_SUB, N_DIMS // M_SUB)          # split the query into 32 chunks of 8 dims

# THE LOOKUP TABLE: distance from each query chunk to each of its 256 centroids.
lookup = ((q_chunks[:, None, :] - codebooks) ** 2).sum(axis=2)
print("lookup table shape:", lookup.shape, "-> built once per query")

In [ ]:
# Now score all 200,000 tickets with lookups and adds only.
t0 = time.perf_counter()
approx_distance = lookup[np.arange(M_SUB), codes].sum(axis=1)
lookup_ms = (time.perf_counter() - t0) * 1000

# Compare against the true distances.
true_distance = ((tickets - q) ** 2).sum(axis=1)

manual_top = np.argsort(approx_distance)[:K]
faiss_top = pq.search(q[None], K)[1][0]

print(f"scored {N_TICKETS:,} tickets using only table lookups + adds")
print(f"our hand-built ADC agrees with faiss on {len(set(manual_top) & set(faiss_top))}/{K} results")
print(f"correlation(approx distance, true distance) = {np.corrcoef(approx_distance, true_distance)[0,1]:.4f}")

pq_search_ms = [r["ms"] for r in pq_sweep if r["m"] == M_SUB][0]
print(f"\nour numpy version : {lookup_ms:6.1f} ms   <- fancy-indexing overhead, NOT the algorithm")
print(f"faiss, same math  : {pq_search_ms:6.3f} ms   <- same lookups, SIMD over packed bytes")
print("\nDon't read our number as PQ's speed. The point of this cell is that the *math*")
print("is only lookups and adds. faiss shows what that costs when implemented properly.")

In [ ]:
# Approximate vs true distance — tight correlation, but visibly noisy up close.
sub = rng.choice(N_TICKETS, 4000, replace=False)

plt.figure(figsize=(5.6, 5))
plt.scatter(true_distance[sub], approx_distance[sub], s=4, alpha=.25)
lims = [true_distance[sub].min(), true_distance[sub].max()]
plt.plot(lims, lims, "r--", lw=1, label="perfect")
plt.xlabel("true distance")
plt.ylabel("PQ approximate distance")
plt.title("PQ distances: right on average, noisy in the details")
plt.legend()
plt.tight_layout()
plt.show()

print("The scatter around the line IS the recall loss. Zoom into the bottom-left")
print("corner — that's where the top-10 live, and that's where the noise hurts.")

---
# 5. Rerank — the best 3 lines in the whole notebook

> ### Value: recovers almost all of PQ's lost recall for a few hundredths of a millisecond.
> This is the highest effort-to-payoff ratio in the entire stack, and it's **the single most
> common missing piece in home-grown implementations.**

**Mechanism:** retrieve the top-100 by cheap PQ distance → load those 100 **full** vectors →
rescore them exactly → return the true top-10 of those.

100 exact distance computations is nothing. First, let's combine IVF and PQ into the index this
is actually built on.

In [ ]:
faiss.omp_set_num_threads(N_BUILD_THREADS)

# IVF-PQ: go to the right shelves (IVF), read compressed summaries once there (PQ).
ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatL2(N_DIMS), N_DIMS, NLIST, M_SUB, N_BITS)
ivfpq.cp.niter = 6

t0 = time.perf_counter()
ivfpq.train(TRAIN_SAMPLE)
ivfpq.add(tickets)
print(f"IVF-PQ built in {time.perf_counter()-t0:.1f}s")

faiss.omp_set_num_threads(1)

ivfpq.nprobe = 16
MS_IVFPQ, REC_IVFPQ = bench(lambda Q: ivfpq.search(Q, K)[1], label="IVF-PQ, no rerank")

In [ ]:
def rerank_search(Q, k=K, n_candidates=100):
    """Shortlist with cheap PQ distances, then rescore the shortlist exactly."""
    _, candidates = ivfpq.search(Q, n_candidates)     # step 1: cheap, approximate
    out = np.empty((len(Q), k), dtype=np.int64)

    for i in range(len(Q)):
        cand = candidates[i]
        cand = cand[cand >= 0]                        # faiss pads short lists with -1
        exact_scores = tickets[cand] @ Q[i]           # step 2: exact, on 100 vectors only
        out[i] = cand[np.argsort(-exact_scores)[:k]]

    return out

print(f"{'candidates':>11} {'ms/query':>10} {'recall@10':>11}   gain vs no-rerank")
print("-" * 56)
for n_cand in [10, 20, 50, 100, 200]:
    ms, rec = bench(lambda Q: rerank_search(Q, n_candidates=n_cand), n=500)
    print(f"{n_cand:>11} {ms:>10.3f} {rec:>11.3f}   {rec-REC_IVFPQ:+.3f}")

MS_RERANK, REC_RERANK = bench(lambda Q: rerank_search(Q, n_candidates=100), n=500)

In [ ]:
gain = REC_RERANK - REC_IVFPQ
cost = MS_RERANK - MS_IVFPQ
print(f"IVF-PQ alone         : recall {REC_IVFPQ:.3f}  at {MS_IVFPQ:.3f} ms")
print(f"IVF-PQ + rerank 100  : recall {REC_RERANK:.3f}  at {MS_RERANK:.3f} ms")
print()
print(f"==> {gain:+.3f} recall for {cost:+.3f} ms.")
print("    If the room remembers one implementation detail, make it this one.")

### The honest catch — and this is what earns you credibility

**Rerank needs the full vectors, which you just spent all that effort compressing away.**
So where do they live?

| Where | Trade-off |
|---|---|
| **On SSD** | Read ~100 vectors per query. Keeps the memory win. Usually the right answer. |
| **In RAM** (faiss `IndexRefineFlat`) | Fastest — but you just gave back the entire 32× saving. |
| **Skip rerank, encode better** (**OPQ**) | A learned rotation applied before PQ that decorrelates dimensions so chunks quantize better. Free at query time. |

Pointing out that the fix has its own cost is exactly what makes the rest of your numbers
believable. Here's OPQ measured, so you can quote a real number.

In [ ]:
faiss.omp_set_num_threads(N_BUILD_THREADS)

opq = faiss.index_factory(N_DIMS, f"OPQ{M_SUB},PQ{M_SUB}")

# Cap the training iterations so this finishes in ~30s rather than several minutes.
opq_rotation = faiss.downcast_VectorTransform(opq.chain.at(0))
opq_rotation.niter = 8
opq_rotation.niter_pq = 4

t0 = time.perf_counter()
opq.train(np.ascontiguousarray(tickets[:20_000]))
opq.add(tickets)
print(f"OPQ built in {time.perf_counter()-t0:.1f}s")

faiss.omp_set_num_threads(1)

plain_pq_recall = [r["recall"] for r in pq_sweep if r["m"] == M_SUB][0]
_, opq_recall = bench(lambda Q: opq.search(Q, K)[1], n=500, label=f"OPQ{M_SUB} (same 32 bytes)")
print(f"plain PQ{M_SUB} for comparison: recall {plain_pq_recall:.3f}")
print(f"==> OPQ buys {opq_recall - plain_pq_recall:+.3f} recall at identical memory and query cost.")

---
# 6. HNSW — Hierarchical Navigable Small World

> ### Value 1: the best recall-per-millisecond available, when your data fits in RAM.
> ### Value 2: `efSearch` is tunable **per request**, with no rebuild. One index serves several quality tiers.
>
> Value 1 is why HNSW is the default in pgvector, Qdrant, Weaviate, Milvus, and OpenSearch.
> Value 2 is underrated and nothing else in this notebook offers it.

**The travel analogy:** you're in a small town in Japan and want a specific café in Osaka. You
don't consult a street map of the whole country. **Flight → train → walk.** Zoom levels.

*If your audience knows skip lists: "it's a skip list where the base layer is a proximity graph
instead of a sorted list." That lands instantly with systems people.*

**The structure:** multiple graph layers over the same points. Layer 0 holds every vector linked
to ~`M` near neighbours (short hops). Each layer up holds a random subset with long-range links.

**The search, in one breath:** start at the top entry point; greedily walk to whichever neighbour
is closer to the query; when no neighbour improves, drop a layer and repeat. At layer 0, keep a
candidate list of size `efSearch` instead of just the single best.

Roughly **O(log N)** hops.

In [ ]:
M_LINKS = 16              # edges per node
EF_CONSTRUCTION = 100     # search width during build: costs build time only, free at query time

faiss.omp_set_num_threads(N_BUILD_THREADS)

hnsw = faiss.IndexHNSWFlat(N_DIMS, M_LINKS)
hnsw.hnsw.efConstruction = EF_CONSTRUCTION

t0 = time.perf_counter()
hnsw.add(tickets)
HNSW_BUILD_S = time.perf_counter() - t0
print(f"HNSW built in {HNSW_BUILD_S:.1f}s   (M={M_LINKS}, efConstruction={EF_CONSTRUCTION})")
print("Note this is the slowest build in the notebook — that's HNSW's real cost.")

faiss.omp_set_num_threads(1)

In [ ]:
hnsw_sweep = []

for ef in [10, 20, 50, 100, 200, 400]:
    hnsw.hnsw.efSearch = ef
    ms, rec = bench(lambda Q: hnsw.search(Q, K)[1])
    hnsw_sweep.append({"ef": ef, "ms": ms, "recall": rec})

print(f"{'efSearch':>9} {'ms/query':>10} {'recall@10':>11}   plausible use case")
print("-" * 62)
use_cases = ["type-ahead", "", "standard search", "", "RAG / agent context", "offline eval"]
for r, note in zip(hnsw_sweep, use_cases):
    print(f"{r['ef']:>9} {r['ms']:>10.3f} {r['recall']:>11.3f}   {note}")

**That table is Value 2 made concrete.** Same index, same box, no rebuild — a **per-request
parameter**. Your latency-sensitive type-ahead and your quality-critical RAG pipeline are served
by one deployment.

IVF's `nprobe` gives you the same freedom — say so — but people rarely realise either one is a
*runtime* knob rather than a config-file setting.

### Look at the actual graph

faiss exposes HNSW's internals, so we can build a small 2D index and **draw the layers**. This is
the slide that makes the structure click.

In [ ]:
# A small 2D index purely so the graph is drawable.
pts_g = rng.normal(size=(500, 2)).astype(np.float32)

graph_index = faiss.IndexHNSWFlat(2, 8)
graph_index.hnsw.efConstruction = 60
graph_index.add(pts_g)

h = graph_index.hnsw
levels = faiss.vector_to_array(h.levels)               # how tall each node is
offsets = faiss.vector_to_array(h.offsets)             # where each node's edges start
neighbors = faiss.vector_to_array(h.neighbors)         # the flat edge array
cum_per_level = faiss.vector_to_array(h.cum_nneighbor_per_level)

def neighbors_of(node, level):
    """Neighbour ids of `node` at a given layer."""
    start = int(offsets[node]) + int(cum_per_level[level])
    end = int(offsets[node]) + int(cum_per_level[level + 1])
    linked = neighbors[start:end]
    return linked[linked >= 0]                         # -1 marks an unused slot

top_level = int(levels.max())
print("nodes present at each level:", np.bincount(levels))
print("node 0's layer-0 neighbours:", neighbors_of(0, 0))

In [ ]:
n_layers = top_level
fig, axes = plt.subplots(1, n_layers, figsize=(4.6 * n_layers, 4.4))
axes = np.atleast_1d(axes)

for level in range(n_layers):
    ax = axes[n_layers - 1 - level]                    # draw top layer leftmost
    members = np.where(levels > level)[0]

    edge_count = 0
    for node in members:
        for nb in neighbors_of(node, level):
            ax.plot([pts_g[node, 0], pts_g[nb, 0]],
                    [pts_g[node, 1], pts_g[nb, 1]],
                    lw=.35, c="steelblue", alpha=.55)
            edge_count += 1

    ax.scatter(pts_g[members, 0], pts_g[members, 1], s=14, c="crimson", zorder=3)
    ax.set_title(f"Layer {level} — {len(members)} nodes, {edge_count} edges")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("HNSW: sparse long-range layers on top, dense short-range graph at the bottom")
plt.tight_layout()
plt.show()

**Read it right to left** (the way a search runs): start in the sparse top layer where a few hops
cross the whole space, then descend into progressively denser layers that refine the position.
That's the flight → train → walk picture, drawn from a real index.

### O(log N) vs O(N), measured

The strongest single argument for HNSW isn't its latency at one size — it's the **shape** of the
curve as the corpus grows.

In [ ]:
faiss.omp_set_num_threads(N_BUILD_THREADS)
scaling = []

for n in [12_500, 25_000, 50_000, 100_000, N_TICKETS]:
    subset = np.ascontiguousarray(tickets[:n])

    # exact, one query at a time
    faiss.omp_set_num_threads(1)
    t0 = time.perf_counter()
    for i in range(50):
        s = subset @ queries[i]
        np.argpartition(-s, K)[:K]
    exact_ms = (time.perf_counter() - t0) / 50 * 1000

    # hnsw
    faiss.omp_set_num_threads(N_BUILD_THREADS)
    hi = faiss.IndexHNSWFlat(N_DIMS, M_LINKS)
    hi.hnsw.efConstruction = EF_CONSTRUCTION
    hi.add(subset)

    faiss.omp_set_num_threads(1)
    hi.hnsw.efSearch = 100
    t0 = time.perf_counter()
    hi.search(queries[:300], K)
    hnsw_ms = (time.perf_counter() - t0) / 300 * 1000

    scaling.append({"n": n, "exact": exact_ms, "hnsw": hnsw_ms})
    print(f"N={n:>8,}   exact {exact_ms:7.2f} ms   hnsw {hnsw_ms:6.3f} ms   "
          f"speedup {exact_ms/hnsw_ms:5.0f}x")
    del hi, subset

faiss.omp_set_num_threads(1)

In [ ]:
ns = [r["n"] for r in scaling]
ex = [r["exact"] for r in scaling]
hn = [r["hnsw"] for r in scaling]

growth_exact = ex[-1] / ex[0]
growth_hnsw = hn[-1] / hn[0]
data_growth = ns[-1] / ns[0]

plt.figure(figsize=(6.4, 4.2))
plt.plot(ns, ex, marker="o", label="exact kNN  (linear)")
plt.plot(ns, hn, marker="s", label="HNSW  (logarithmic)")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("corpus size")
plt.ylabel("ms per query")
plt.title("The shape of the curve is the argument")
plt.grid(alpha=.3, which="both")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Corpus grew {data_growth:.0f}x  ->  exact got {growth_exact:.1f}x slower,"
      f" HNSW got {growth_hnsw:.1f}x slower.")

### Where HNSW's value runs out

- **Memory.** HNSW stores the full vectors **plus** the graph. Measured below — compare it to
  IVF-PQ's footprint and the gap is the whole trade: **HNSW buys speed with RAM; IVF-PQ buys
  RAM with accuracy.**
- **Deletes are genuinely awkward.** You can't cleanly remove a node without risking
  disconnecting the graph, so implementations use tombstones and rebuild periodically. For
  high-churn data (sessions, carts, ephemeral docs) that's a real operational cost.
- **Build time.** You measured it above. At 10M vectors it's hours.

In [ ]:
graph_bytes = N_TICKETS * M_LINKS * 2 * 4     # ~2M links at layer 0, 4-byte ids (approximate)
hnsw_bytes = raw_bytes + graph_bytes

print(f"{'index':<22} {'memory':>12}")
print("-" * 36)
print(f"{'raw vectors':<22} {raw_bytes/1e6:>9.1f} MB")
print(f"{'HNSW (vectors+graph)':<22} {hnsw_bytes/1e6:>9.1f} MB")
print(f"{'IVF-PQ (codes)':<22} {pq_bytes/1e6:>9.1f} MB")
print()
print(f"HNSW needs {hnsw_bytes/pq_bytes:.0f}x more memory than IVF-PQ.")

---
# 7. The value ledger

Every technique, on the same problem, with the numbers this notebook just measured.
**This is your closing slide.**

In [ ]:
ivf16 = [r for r in ivf_sweep if r["nprobe"] == 16][0]
hnsw100 = [r for r in hnsw_sweep if r["ef"] == 100][0]

ivf_work_saving = N_TICKETS / ivf16["scanned"]

ledger = [
    ("Exact (numpy)",        EXACT_MS,        raw_bytes,  1.0,            "correctness + ground truth"),
    ("+ IVF (nprobe=16)",    ivf16["ms"],     raw_bytes,  ivf16["recall"], f"{ivf_work_saving:.0f}x less data scanned"),
    ("+ PQ (IVF-PQ)",        MS_IVFPQ,        pq_bytes,   REC_IVFPQ,       f"{raw_bytes/pq_bytes:.0f}x less memory"),
    ("+ rerank (top-100)",   MS_RERANK,       pq_bytes,   REC_RERANK,      "recall bought back, ~free"),
    ("HNSW (efSearch=100)",  hnsw100["ms"],   hnsw_bytes, hnsw100["recall"], "best recall/latency - for RAM"),
]

print(f"{'step':<22} {'ms/query':>9} {'speedup':>8} {'memory':>11} {'recall':>8} {'cores@200qps':>13}  what it bought")
print("-" * 118)
for name, ms, mem, rec, note in ledger:
    print(f"{name:<22} {ms:>9.3f} {EXACT_MS/ms:>7.0f}x {mem/1e6:>8.1f} MB {rec:>8.3f}"
          f" {TARGET_QPS*ms/1000:>13.2f}  {note}")

In [ ]:
# The ann-benchmarks-style chart: recall vs throughput. Up and to the right wins.
plt.figure(figsize=(7.4, 5))

qps_ivf = [1000 / r["ms"] for r in ivf_sweep]
rec_ivf = [r["recall"] for r in ivf_sweep]
plt.plot(qps_ivf, rec_ivf, marker="o", label="IVF (sweeping nprobe)")

qps_hnsw = [1000 / r["ms"] for r in hnsw_sweep]
rec_hnsw = [r["recall"] for r in hnsw_sweep]
plt.plot(qps_hnsw, rec_hnsw, marker="s", label="HNSW (sweeping efSearch)")

plt.scatter([1000 / MS_IVFPQ], [REC_IVFPQ], marker="v", s=110, c="darkorange",
            zorder=5, label="IVF-PQ (no rerank)")
plt.scatter([1000 / MS_RERANK], [REC_RERANK], marker="*", s=260, c="crimson",
            zorder=5, label="IVF-PQ + rerank")
plt.scatter([1000 / EXACT_MS], [1.0], marker="X", s=140, c="black",
            zorder=5, label="exact")

plt.xscale("log")
plt.xlabel("queries per second per core  (higher = better)")
plt.ylabel("recall@10  (higher = better)")
plt.title("The only chart that matters: up and to the right")
plt.grid(alpha=.3, which="both")
plt.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

### How to choose — read straight down

- **< 100k vectors** → **brute force.** Genuinely. Don't add an index; add three lines of numpy.
- **Fits in RAM, want best quality/latency** → **HNSW.** The default for good reason.
- **RAM is the constraint, or you're past ~100M vectors** → **IVF-PQ + rerank.**
- **Both** → HNSW+PQ, IVF-HNSW (HNSW over the centroids), or **DiskANN** for billion-scale on SSD.
- **Always** → keep exact search around as the ruler.

---
# 8. The gotcha that actually decides your architecture

Everything above assumed the query is *just* a vector. Real queries almost never are:

> "Nearest neighbours **WHERE** `tenant_id = X` **AND** `status = 'open'` **AND** `created_at > Y`."

This is where teams get hurt, and it's usually what determines which vector database you end up
with. Let's break it on purpose.

**The naive approach — post-filtering:** run normal ANN search, then drop results that fail the
predicate. Watch what happens when the predicate is selective.

In [ ]:
# Suppose only 2% of tickets are "priority" — a fairly ordinary filter.
is_priority = rng.random(N_TICKETS) < 0.02
print(f"{is_priority.sum():,} of {N_TICKETS:,} tickets match the filter ({100*is_priority.mean():.1f}%)")

hnsw.hnsw.efSearch = 100
_, found = hnsw.search(queries[:300], K)

kept = np.array([is_priority[row[row >= 0]].sum() for row in found])

print(f"\nPOST-FILTERING (search first, filter after):")
print(f"  asked for {K} results, got on average {kept.mean():.2f}")
print(f"  {100*np.mean(kept == 0):.0f}% of queries returned ZERO results")

**That is a broken product feature, not a tuning problem.** You asked for 10 results and most
users got none — while plenty of matching tickets existed, just outside the top 10.

The fix is to push the predicate **into** the search, so the index keeps walking until it has
enough *matching* results. faiss does this with an `IDSelector`.

In [ ]:
selector = faiss.IDSelectorArray(np.where(is_priority)[0].astype(np.int64))

# IVF with the filter pushed down into the scan.
ivf.nprobe = 64
params = faiss.SearchParametersIVF(sel=selector, nprobe=64)
_, filtered = ivf.search(queries[:300], K, params=params)
ivf_valid = np.mean((filtered >= 0).sum(axis=1))

# HNSW with the same filter pushed into the graph walk.
hnsw.hnsw.efSearch = 200
params_h = faiss.SearchParametersHNSW(sel=selector, efSearch=200)
_, filtered_h = hnsw.search(queries[:300], K, params=params_h)
hnsw_valid = np.mean((filtered_h >= 0).sum(axis=1))

print(f"{'strategy':<34} {'avg results out of 10':>22}")
print("-" * 58)
print(f"{'post-filter (search then drop)':<34} {kept.mean():>22.2f}")
print(f"{'IVF + IDSelector (pushed down)':<34} {ivf_valid:>22.2f}")
print(f"{'HNSW + IDSelector (pushed down)':<34} {hnsw_valid:>22.2f}")

**Filtered search deserves its own talk** — and if you give this one, offer it. The short version
of what's left out:

- **Pre-filtering** (materialise matching ids, then brute-force them) wins when the filter is
  *very* selective — if only 500 tickets match, just scan those 500.
- **Filtered graph traversal** degrades when the filter is selective *and* the matching points are
  scattered: the walk spends its whole budget stepping through rejected nodes, and can get stuck.
- **Partitioned indexes** (one index per tenant) sidestep all of it, at the cost of many small
  indexes and worse memory locality.
- Every vector database answers this differently, and their marketing pages mostly don't say how.
  **Benchmark it with your real filter selectivity before you pick one.**

---
# 9. Things to try (the useful part after the talk)

These are ordered so each one teaches something specific.

1. **Break IVF on purpose.** Set `NLIST = 16` and re-run section 3. Buckets get huge, and
   `nprobe=1` scans 6% of the corpus. Now set `NLIST = 8192` — buckets get tiny and you need a
   much larger `nprobe` for the same recall. **There's a sweet spot near sqrt(N), and now you've
   felt both sides of it.**

2. **Make the data hostile.** Set `N_TOPICS = 1` and re-run everything. With no cluster structure,
   every method collapses. This is the curse of dimensionality in its purest form — and it's why
   benchmarking on random vectors will make you reject techniques that work fine on real data.

3. **Find PQ's cliff for your data.** In the `m` sweep, add `m = 8`. Then set `ALPHA = 0.3`
   (a flatter spectrum, less redundancy to exploit) and re-run. **Watch every PQ recall number
   drop.** That's the connection between "my embeddings are correlated" and "PQ works."

4. **Prove `efConstruction` is free at query time.** Rebuild HNSW with `EF_CONSTRUCTION = 400`.
   Build time goes up a lot; query latency doesn't move; recall improves. It's the one knob you
   should just be generous with.

5. **Measure the rerank sweet spot.** In section 5, sweep `n_candidates` up to 1000. Find where
   the recall gain stops paying for the latency. It's usually far lower than people guess.

6. **Simulate drift.** Train IVF on `tickets[:50_000]` only, then add all of them. Recall at a
   fixed `nprobe` drops, because the centroids no longer describe the corpus. **That's what a
   stale index looks like in production** — no error, no alert, just quietly worse results.

7. **Swap in real embeddings.** `pip install sentence-transformers`, encode a few thousand real
   documents, and re-run. Compare the eigenvalue spectrum plot in section 0 against the synthetic
   one — that comparison alone is worth the exercise.

---
## Reference: the whole talk in one table

| Technique | Value it adds | Its dial | Where it runs out |
|---|---|---|---|
| **Exact kNN** | Correctness; the ruler for everything else | — | Linear cost; hopeless past a few hundred thousand |
| **IVF** | Skip ~99% of the data | `nprobe` | Memory unchanged; needs training; drifts |
| **PQ** | ~32× less memory; cheaper comparisons too | `m` (bytes/vector) | Noisy ranking; has a cliff, not a slope |
| **Rerank** | Buys back nearly all lost recall for ~free | `n_candidates` | Needs the full vectors somewhere |
| **OPQ** | Better PQ at identical cost | — | Slower to train |
| **HNSW** | Best recall/latency; per-query quality dial | `efSearch` | RAM-hungry; awkward deletes; slow builds |

**The one-sentence comparison:** *HNSW buys speed with RAM; IVF-PQ buys RAM with accuracy;
rerank buys the accuracy back.*